In [0]:
# %pip install -U -qqqq 
# backoff 
# databricks-langchain 
# langgraph==0.5.3 
# uv 
# databricks-agents 
# mlflow-skinny[databricks] 
# chromadb 
# sentence-transformers 
# langchain-huggingface
# langchain-chroma 
# wikipedia 
# faiss-cpu

In [0]:
%pip install -U -q databricks-langchain langchain==0.3.7 faiss-cpu wikipedia langchain-community chromadb langchain-openai tiktoken


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA

### Simple RAG

In [0]:
# Retriever Config
MAX_WIKI_DOCS_PER_TOPIC = 10 #TODO: recommend starting with a smaller number for testing purposes
VECTOR_TOP_K = 5 # number of documents to return
EMBEDDING_MODEL = "databricks-bge-large-en" # Embedding model endpoint name

# LLM Config
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-1-8b-instruct"

# Initialize embeddings + LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.2)


In [0]:
from langchain.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="deep learning", load_max_docs=5)
docs = loader.load()

In [0]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
print(f"len chunk {len(chunks)} ")

len chunk 68 


In [0]:
from langchain_community.vectorstores import FAISS

embeddings = DatabricksEmbeddings(endpoint=EMBEDDING_MODEL)

# Build FAISS index from your chunks
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [0]:
# Initialize embeddings + LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # 检索Top-3相关片段

In [0]:
prompt = ChatPromptTemplate.from_template("""
Please answer the question based on the following context information.
If the context does not provide relevant information, please directly say:
"Based on the available information, I cannot answer this question."

Context: {context}

Question: {question}

Answer:
""")

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # The simplest chain type: stuff all retrieved context into the prompt
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

# 4. Test query
query = "what is deep learning?"
result = rag_chain.invoke({"query": query})

print(f"Question: {query}")
print(f"Answer: {result['result']}")

Question: what is deep learning?
Answer: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network, using a hierarchy of layers to transform input data into a progressively more abstract and composite representation.
